# HR PEOPLE ANALYTICS — Partie 2 : Feature Engineering 

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings("ignore")

## Chargement du master dataset

In [13]:
master = pd.read_csv("master_hr_dataset.csv", parse_dates=["StartDate", "ExitDate", "DOB"])
print(f"Dataset chargé : {master.shape}")

Dataset chargé : (3000, 36)


## Variables liées au statut

In [14]:
# Employé encore actif ou non
master["Is_Active"] = master["ExitDate"].isna()

# Termination propre
master["TerminationDescription"] = master["TerminationDescription"].fillna("Still Employed")
master["TerminationType"] = master["TerminationType"].fillna("Still Employed")

# Durée d'emploi en années (pour les partis : StartDate → ExitDate / pour les actifs : jusqu'à aujourd'hui)
today = pd.Timestamp.today()
master["Employment_Duration"] = np.where(
    master["Is_Active"],
    (today - master["StartDate"]).dt.days / 365.25,
    (master["ExitDate"] - master["StartDate"]).dt.days / 365.25
).round(2)

## Variables temporelles

In [15]:
master["Start_Year"]    = master["StartDate"].dt.year
master["Start_Month"]   = master["StartDate"].dt.month
master["Start_Quarter"] = master["StartDate"].dt.quarter

# Saison d'embauche
def get_season(month):
    if month in [12, 1, 2]:  return "Winter"
    elif month in [3, 4, 5]: return "Spring"
    elif month in [6, 7, 8]: return "Summer"
    else:                    return "Fall"

master["Hiring_Season"] = master["Start_Month"].apply(get_season)

## Variables d'âge et de génération

In [16]:
master["DOB"] = pd.to_datetime(master["DOB"], errors="coerce")
master["Age"] = ((today - master["DOB"]).dt.days / 365.25).round(1)

# Génération
def get_generation(birth_year):
    if   birth_year >= 1997: return "Gen Z"
    elif birth_year >= 1981: return "Millennial"
    elif birth_year >= 1965: return "Gen X"
    else:                    return "Boomer"

master["Generation"] = master["DOB"].dt.year.apply(
    lambda y: get_generation(y) if pd.notna(y) else "Unknown"
)

# Tranche d'âge
master["Age_Band"] = pd.cut(
    master["Age"],
    bins=[0, 30, 40, 50, 60, 100],
    labels=["<30", "30-40", "40-50", "50-60", "60+"]
)

## Variables d'engagement & performance

In [17]:
# Score composite d'engagement (moyenne pondérée)
master["Engagement_Composite"] = (
    master["Engagement_Score"]   * 0.4 +
    master["Satisfaction_Score"] * 0.35 +
    master["WorkLife_Score"]     * 0.25
).round(3)

# Flag : employé à risque de départ (faible engagement + faible perf)
perf_map = {
    "PIP": 1,
    "Needs Improvement": 2,
    "Fully Meets": 3,
    "Exceeds": 4
}
master["Performance_Num"] = master["Performance Score"].map(perf_map)

master["Attrition_Risk_Flag"] = (
    (master["Engagement_Composite"] < master["Engagement_Composite"].quantile(0.25)) &
    (master["Performance_Num"] <= 2)
).astype(int)

print(f"Employés à risque détectés : {master['Attrition_Risk_Flag'].sum()}")

Employés à risque détectés : 70


## Variables de formation

In [18]:
# Investissement formation par employé (coût / durée)
master["Training_Cost_Per_Day"] = (
    master["Total_Training_Cost"] / master["Avg_Training_Days"]
).replace([np.inf, -np.inf], np.nan).round(2)

# Flag : employé peu formé (moins de formations que la médiane)
median_training = master["Training_Count"].median()
master["Low_Training_Flag"] = (master["Training_Count"] < median_training).astype(int)

## Encodage des variables catégorielles

In [19]:
# Label encoding pour les variables ordinales
master["PayZone_Num"] = master["PayZone"].map({"ZONE A": 1, "ZONE B": 2, "ZONE C": 3})

# One-hot encoding pour les variables nominales
cat_cols = ["GenderCode", "RaceDesc", "DepartmentType", "EmployeeType", "Generation", "Hiring_Season"]
master = pd.get_dummies(master, columns=cat_cols, drop_first=False, dtype=int)

print(f"Après encodage : {master.shape[1]} features")

Après encodage : 68 features


## Normalisation des variales numériques

In [20]:
num_cols = [
    "Age", "Seniority", "Employment_Duration",
    "Engagement_Composite", "Training_Count",
    "Total_Training_Cost", "Training_Pass_Rate"
]

scaler = StandardScaler()
scaled = scaler.fit_transform(master[num_cols].fillna(0))
scaled_df = pd.DataFrame(scaled, columns=[f"{c}_scaled" for c in num_cols], index=master.index)
master = pd.concat([master, scaled_df], axis=1)

## Export

In [22]:
master.to_csv("master_hr_features.csv", index=False)
print(f"\nmaster_hr_features.csv exporté : {master.shape[0]} lignes × {master.shape[1]} colonnes")

# Résumé des features créées
new_features = [
    "Is_Active", "Employment_Duration", "Start_Year", "Start_Month",
    "Start_Quarter", "Hiring_Season", "Generation", "Age_Band",
    "Engagement_Composite", "Performance_Num", "Attrition_Risk_Flag",
    "Training_Cost_Per_Day", "Low_Training_Flag", "PayZone_Num"
]
print(f"\nNouvelles features créées ({len(new_features)}) :")
for f in new_features:
    print(f"   → {f}")


master_hr_features.csv exporté : 3000 lignes × 75 colonnes

Nouvelles features créées (14) :
   → Is_Active
   → Employment_Duration
   → Start_Year
   → Start_Month
   → Start_Quarter
   → Hiring_Season
   → Generation
   → Age_Band
   → Engagement_Composite
   → Performance_Num
   → Attrition_Risk_Flag
   → Training_Cost_Per_Day
   → Low_Training_Flag
   → PayZone_Num
